# Phase 4, session B: NeRF-synthetic with gsplat's rasterizer (reference) + diagnostic

The same 8 scenes, same training loop and settings, with gsplat's rasterizer (bench/train_gsplat.py), for the measured mine-vs-gsplat comparison. Then the alpha-clamp diagnostic on the trained lego.

Run on **Kaggle** with **GPU T4 x2** and Internet on: **Save Version → Save & Run All (Commit)**, then stop the draft session. With one GPU (Colab) the jobs run one after the other.

Every run: 30k iterations, paper hyperparameters, seed 0, the paper's evaluation protocol. Mip-NeRF 360 / T&T runs cap the Gaussian count at 3,000,000 to fit the T4's 15 GB (recorded in each results file). Checkpoints every 1000 iterations and a stop at 10.5 h: if the session ends early, attach this version's output as an input and run it again to resume.

**GPU-hour estimate (from the Phase-3 timings; a guess until measured):** gsplat build ~0.2 h; ~0.3-0.6 h per scene, 4 per GPU: about 1.5-3 h.

At the end: download `phase4_B_results.zip` from the Output tab.

In [ ]:
import os, sys, subprocess, shutil, glob, time
KAGGLE = os.path.exists("/kaggle/working")
COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if KAGGLE else ("/content" if COLAB else os.path.abspath("."))
DATA = "/tmp/data" if KAGGLE else os.path.join(WORK, "data")   # outside /kaggle/working: not saved as output
REPO_URL = "https://github.com/GARVIT-bit/gaussian-splatting.git"
BRANCH = "main"
REPO = os.path.join(WORK, "gaussian-splatting")
print("platform:", "kaggle" if KAGGLE else "colab" if COLAB else "local", "| work dir:", WORK)

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

if not os.path.exists(REPO):
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
else:
    sh(f"git -C {REPO} pull --ff-only")
sh(f"git -C {REPO} log -1 --format='%h %s'")
sh("pip install -q lpips pytest")   # torch, numpy, pillow are preinstalled on Kaggle and Colab
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
os.makedirs(DATA, exist_ok=True)

In [ ]:
SESSION = "B"
CAP = 3_000_000
JOBS = {
 "0": [
  {
   "name": "lego_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "lego"
  },
  {
   "name": "ship_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "ship"
  },
  {
   "name": "drums_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "drums"
  },
  {
   "name": "materials_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "materials"
  }
 ],
 "1": [
  {
   "name": "chair_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "chair"
  },
  {
   "name": "ficus_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "ficus"
  },
  {
   "name": "hotdog_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "hotdog"
  },
  {
   "name": "mic_gsplat",
   "who": "gsplat",
   "dataset": "blender",
   "scene": "mic"
  }
 ]
}
JOBS = {int(k): v for k, v in JOBS.items()}

In [ ]:
# all unit tests + GPU tests (Triton kernels vs the reference); stops the notebook if anything fails
sh(f"cd {REPO} && python -m pytest -p no:cacheprovider -rs tests")

In [ ]:
# datasets (into /tmp/data, not saved as output)
sh(f"cd {REPO} && python scripts/get_data.py blender --dest {DATA} --scenes chair drums ficus hotdog lego materials mic ship")

In [ ]:
# gsplat: reference only (imported by bench/train_gsplat.py and bench/compare_gsplat.py, never by my code).
r = subprocess.run("pip install -q gsplat && python -c \"import torch, gsplat; from gsplat import rasterization; "
                   "m=torch.zeros(1,3,device='cuda'); q=torch.tensor([[1.,0,0,0]],device='cuda'); "
                   "s=torch.full((1,3),.1,device='cuda'); o=torch.ones(1,device='cuda'); c=torch.ones(1,3,device='cuda'); "
                   "V=torch.eye(4,device='cuda'); V[2,3]=3; K=torch.tensor([[50.,0,32],[0,50,32],[0,0,1]],device='cuda'); "
                   "print('gsplat', gsplat.__version__, rasterization(m,q,s,o,c,V[None],K[None],64,64)[0].shape)\"",
                   shell=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError("gsplat could not be installed/built in this session: the gsplat runs cannot happen")

## Training jobs

In [ ]:
RUNS = os.path.join(WORK, "runs")
RES = os.path.join(WORK, f"phase4_{SESSION}_results")
os.makedirs(RUNS, exist_ok=True)
os.makedirs(RES, exist_ok=True)

def job_cmd(j):
    ds, scene = j["dataset"], j["scene"]
    if ds == "blender":
        data = f"--data {DATA}/nerf_synthetic/{scene} --format blender --downscale 1"
    elif ds == "mipnerf360":
        folder = "images_4" if scene in ("bicycle", "flowers", "garden", "stump", "treehill") else "images_2"
        data = f"--data {DATA}/360_v2/{scene} --format colmap --images {folder} --max-gaussians {CAP}"
    else:
        data = f"--data {DATA}/tandt/{scene} --format colmap --images images --max-gaussians {CAP}"
    script = "scripts/train.py --render-mode triton" if j["who"] == "mine" else "bench/train_gsplat.py"
    name = j["name"]
    return (f"python {script} {data} --iterations 30000 --ckpt-every 1000 --log-every 500 --time-limit-hours 10.5 "
            f"--out {RUNS}/{name} --name {name} --results {RES}/{name}.md")

# resume: bring back checkpoints from an earlier version's output, if attached as an input
for jobs in JOBS.values():
    for j in jobs:
        prev = glob.glob(f"/kaggle/input/**/runs/{j['name']}/ckpt.pt", recursive=True)
        dst = os.path.join(RUNS, j["name"])
        if prev and not os.path.exists(os.path.join(dst, "ckpt.pt")):
            shutil.copytree(os.path.dirname(prev[0]), dst, dirs_exist_ok=True)
            print("resuming", j["name"], "from", prev[0])

import torch
n_gpu = torch.cuda.device_count()
print("GPUs:", n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
chains = {g: " ; ".join(f"({job_cmd(j)})" for j in jobs) for g, jobs in JOBS.items()}
if n_gpu < 2:   # Colab: everything on one GPU, one job after the other
    chains = {0: " ; ".join(chains.values())}
procs = {}
for gpu, cmd in chains.items():
    log = open(os.path.join(RES, f"gpu{gpu}.log"), "w")
    print(f"GPU {gpu}:", cmd.replace(" ; ", "\n   "), "\n")
    procs[gpu] = subprocess.Popen(f"cd {REPO} && {cmd}", shell=True, stdout=log, stderr=subprocess.STDOUT,
                                  env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    tails = {g: open(os.path.join(RES, f"gpu{g}.log")).read().strip().splitlines()[-1:] for g in procs}
    print(f"[{(time.time() - t0) / 3600:.2f} h]", {g: t[0][:110] if t else "" for g, t in tails.items()}, flush=True)
for g, p in procs.items():
    print(f"GPU {g} exit code {p.returncode}")
    print("".join(open(os.path.join(RES, f"gpu{g}.log")).readlines()[-15:]))

## Diagnostic: my rasterizer vs gsplat on a trained scene (alpha clamp 0.99 vs 0.999)

In [ ]:
sh(f"cd {REPO} && python bench/compare_gsplat.py --scene {RUNS}/lego_gsplat/final.pt --data {DATA}/nerf_synthetic/lego "
   f"--views 20 --out {RES}/compare_gsplat_lego.md")

## Package results

In [ ]:
for jobs in JOBS.values():
    for j in jobs:
        d = os.path.join(RUNS, j["name"])
        for f in glob.glob(os.path.join(d, "log.csv")) + glob.glob(os.path.join(d, "test_000.png")):
            shutil.copy(f, os.path.join(RES, f"{j['name']}_{os.path.basename(f)}"))
sh(f"cd {REPO} && python bench/phase4_tables.py --runs {RES} --out {RES}/phase4_summary_{SESSION}.md")
shutil.make_archive(RES, "zip", RES)
print("download:", RES + ".zip   (final.pt files stay in runs/ in this version's output, for the viewer later)")